# 05 — BERT Fine-tuning (Google Colab)

**Objetivo:** Treinar os 3 modelos BERT (M4a BERTimbau, M4b FinBERT, M4c FinBERT-PT-BR) para classificacao binaria `mercado` vs `outros`, utilizando GPU do Google Colab.

**Pre-requisitos:**
- Notebook 01 executado localmente (splits gerados)
- Arquivo `colab_splits.zip` gerado via `uv run python scripts/colab_pack.py`
- Arquivo zip enviado ao Google Drive

**Saidas:**
- Modelos treinados para cada variante BERT
- Predicoes no split de validacao e teste (CSV padrao)
- Metricas de avaliacao (JSON)
- Arquivo `colab_bert_results.zip` para download

**Tempo estimado:** ~2-4 horas (GPU T4)

## 0. Verificacao de GPU e setup do ambiente

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU nao detectada. Va em Runtime > Change runtime type > GPU (T4)."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# ---------- Configuracao ----------
# URL HTTPS do repositorio no GitHub (ajuste se necessario)
REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
REPO_BRANCH = "main"

# Pasta no Google Drive onde esta o colab_splits.zip
# e onde os resultados serao salvos
DRIVE_FOLDER = "economy-classifier"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"Google Drive montado. Pasta de trabalho: {DRIVE_BASE}")

In [ ]:
import subprocess, os

REPO_DIR = Path("/content/economy-classifier")

if REPO_DIR.exists():
    print(f"Repositorio ja existe em {REPO_DIR}, atualizando...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    print(f"Clonando repositorio...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

# Instalar o pacote em modo editavel
subprocess.run(
    ["pip", "install", "-e", str(REPO_DIR), "-q"],
    check=True,
)
print("Pacote economy_classifier instalado.")

## 1. Carga dos splits

O arquivo `colab_splits.zip` deve estar em `Google Drive > economy-classifier/`.

In [ ]:
import zipfile

ZIP_PATH = DRIVE_BASE / "colab_splits.zip"
SPLITS_DIR = REPO_DIR / "artifacts" / "splits"

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo nao encontrado: {ZIP_PATH}\n"
        f"Faca upload do colab_splits.zip para Google Drive > {DRIVE_FOLDER}/"
    )

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for member in zf.namelist():
        filename = Path(member).name
        if filename:
            target = SPLITS_DIR / filename
            target.write_bytes(zf.read(member))
            print(f"  + {filename}")

print(f"\nSplits extraidos em: {SPLITS_DIR}")

In [ ]:
import json
import pandas as pd

balanced_train = pd.read_parquet(SPLITS_DIR / "balanced_train.parquet")
val = pd.read_parquet(SPLITS_DIR / "val.parquet")
test = pd.read_parquet(SPLITS_DIR / "test.parquet")
split_meta = json.loads((SPLITS_DIR / "split_metadata.json").read_text())

print(f"Treino balanceado: {len(balanced_train):,} linhas")
print(f"Validacao:         {len(val):,} linhas")
print(f"Teste:             {len(test):,} linhas")
print(f"Seed original:     {split_meta['seed']}")
print(f"Corpus SHA256:     {split_meta['corpus_sha256'][:16]}...")

## 2. Funcoes auxiliares

Imports do pacote e funcao de treino completa (treino + predicoes val/teste + persistencia).

In [ ]:
import time
import numpy as np

from economy_classifier.bert import (
    BertTrainingConfig,
    MODEL_REGISTRY,
    predict_texts,
    train_bert_classifier,
)
from economy_classifier.evaluation import compute_binary_metrics, compute_roc_auc
from economy_classifier.project import (
    build_run_metadata,
    create_run_directory,
    persist_run_artifacts,
)


# --- Configuracao de treino para Colab (GPU T4, 16 GB VRAM) ---
COLAB_TRAIN_BATCH = 8
COLAB_GRAD_ACCUM = 2        # batch efetivo = 16
COLAB_EVAL_BATCH = 32
COLAB_EPOCHS = 3
COLAB_PATIENCE = 1
COLAB_SAVE_LIMIT = 2


def train_and_evaluate_bert(
    variant_key: str,
    balanced_train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    """Treina um modelo BERT, gera predicoes em val e test, persiste artefatos.

    Returns dict com run_dir, metrics, predictions_val, predictions_test.
    """
    model_name = MODEL_REGISTRY[variant_key]
    print(f"\n{'='*60}")
    print(f"  {variant_key.upper()} — {model_name}")
    print(f"{'='*60}\n")

    config = BertTrainingConfig(
        model_name=model_name,
        per_device_train_batch_size=COLAB_TRAIN_BATCH,
        gradient_accumulation_steps=COLAB_GRAD_ACCUM,
        per_device_eval_batch_size=COLAB_EVAL_BATCH,
        num_train_epochs=COLAB_EPOCHS,
        early_stopping_patience=COLAB_PATIENCE,
        save_total_limit=COLAB_SAVE_LIMIT,
    )

    run_dir = create_run_directory("bert-training", variant_key)
    print(f"Run dir: {run_dir}\n")

    # --- Treino ---
    t0 = time.perf_counter()
    result = train_bert_classifier(
        balanced_train_df, val_df, run_dir=run_dir, config=config,
    )
    total_time = time.perf_counter() - t0
    print(f"\nTreino concluido em {total_time/60:.1f} min")

    # --- Metricas de validacao ---
    preds_val = result["predictions"]
    val_metrics = compute_binary_metrics(
        preds_val["y_true"].values, preds_val["y_pred"].values,
    )
    val_metrics["auc_roc"] = round(
        compute_roc_auc(preds_val["y_true"].values, preds_val["y_score"].values), 4,
    )
    print(f"\nValidacao — F1: {val_metrics['f1']:.4f}  "
          f"AUC: {val_metrics['auc_roc']:.4f}")

    # --- Predicoes no teste ---
    print(f"\nGerando predicoes no teste ({len(test_df):,} amostras)...")
    t0 = time.perf_counter()
    test_preds_raw = predict_texts(
        test_df["text"].fillna("").tolist(),
        model_dir=result["model_dir"],
        method=variant_key,
        batch_size=COLAB_EVAL_BATCH,
    )
    test_time = time.perf_counter() - t0
    print(f"Inferencia no teste concluida em {test_time:.1f}s")

    preds_test = pd.DataFrame({
        "index": test_df.index.tolist(),
        "y_true": test_df["label"].tolist(),
        "y_pred": test_preds_raw["y_pred"].tolist(),
        "y_score": test_preds_raw["y_score"].tolist(),
        "method": variant_key,
    })

    # --- Persistencia ---
    preds_val.to_csv(run_dir / "predictions_val.csv", index=False)
    preds_test.to_csv(run_dir / "predictions_test.csv", index=False)

    metadata = build_run_metadata(
        run_dir=run_dir,
        stage="bert-training",
        parameters=config.to_dict(),
        inputs={
            "balanced_train_rows": len(balanced_train_df),
            "val_rows": len(val_df),
            "test_rows": len(test_df),
            "corpus_sha256": split_meta.get("corpus_sha256", "unknown"),
        },
        outputs={
            "model_dir": str(result["model_dir"]),
            "predictions_val": str(run_dir / "predictions_val.csv"),
            "predictions_test": str(run_dir / "predictions_test.csv"),
        },
        summary={
            "variant": variant_key,
            "model_name": model_name,
            "val_metrics": val_metrics,
            "trainer_metrics": result["metrics"],
        },
        timing={
            "train_seconds": result["timing"]["train_seconds"],
            "test_inference_seconds": round(test_time, 2),
            "total_seconds": round(total_time + test_time, 2),
        },
    )
    persist_run_artifacts(
        run_dir=run_dir, metadata=metadata, metrics=val_metrics,
    )

    # Copiar resultados para o Drive como backup
    backup_dir = DRIVE_BASE / "results" / run_dir.name
    backup_dir.mkdir(parents=True, exist_ok=True)
    for f in ["predictions_val.csv", "predictions_test.csv", "metrics.json", "run_metadata.json"]:
        src = run_dir / f
        if src.exists():
            (backup_dir / f).write_bytes(src.read_bytes())

    print(f"Backup salvo em Drive: {backup_dir.relative_to('/content/drive/MyDrive')}")

    # Limpar checkpoints para liberar espaco
    checkpoints_dir = run_dir / "checkpoints"
    if checkpoints_dir.exists():
        import shutil
        shutil.rmtree(checkpoints_dir)
        print("Checkpoints intermediarios removidos para liberar espaco.")

    torch.cuda.empty_cache()

    return {
        "variant": variant_key,
        "run_dir": run_dir,
        "val_metrics": val_metrics,
        "predictions_val": preds_val,
        "predictions_test": preds_test,
    }


print("Funcoes carregadas. Pronto para treinar.")

## 3. M4a — BERTimbau

`neuralmind/bert-base-portuguese-cased` — BERT pre-treinado em portugues brasileiro (generalista).

In [ ]:
result_bertimbau = train_and_evaluate_bert("bertimbau", balanced_train, val, test)

## 4. M4b — FinBERT

`ProsusAI/finbert` — BERT pre-treinado em textos financeiros em ingles. Avalia transferencia cross-lingual de dominio.

In [ ]:
result_finbert = train_and_evaluate_bert("finbert", balanced_train, val, test)

## 5. M4c — FinBERT-PT-BR

`lucas-leme/FinBERT-PT-BR` — BERT financeiro adaptado para portugues brasileiro. Combina dominio e idioma.

In [ ]:
result_finbert_ptbr = train_and_evaluate_bert("finbert_ptbr", balanced_train, val, test)

## 6. Resumo comparativo

In [ ]:
all_results = [result_bertimbau, result_finbert, result_finbert_ptbr]

comparison = pd.DataFrame([
    {
        "Metodo": r["variant"],
        "Precision": r["val_metrics"]["precision"],
        "Recall": r["val_metrics"]["recall"],
        "F1": r["val_metrics"]["f1"],
        "Accuracy": r["val_metrics"]["accuracy"],
        "AUC-ROC": r["val_metrics"].get("auc_roc", None),
    }
    for r in all_results
]).sort_values("F1", ascending=False)

print("Metricas de validacao — 3 variantes BERT:\n")
print(comparison.to_string(index=False))

## 7. Empacotar resultados para download

Gera `colab_bert_results.zip` com predicoes, metricas e metadados de todas as variantes. Este arquivo deve ser baixado e descompactado localmente com:

```bash
uv run python scripts/colab_unpack.py colab_bert_results.zip
```

In [ ]:
RESULT_ZIP = DRIVE_BASE / "colab_bert_results.zip"
ARTIFACT_FILES = [
    "predictions_val.csv",
    "predictions_test.csv",
    "metrics.json",
    "run_metadata.json",
]

with zipfile.ZipFile(RESULT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for r in all_results:
        run_dir = r["run_dir"]
        run_name = run_dir.name
        for fname in ARTIFACT_FILES:
            fpath = run_dir / fname
            if fpath.exists():
                zf.write(fpath, f"{run_name}/{fname}")

        # Incluir modelo treinado (tokenizer + weights)
        model_dir = run_dir / "model"
        if model_dir.exists():
            for model_file in model_dir.rglob("*"):
                if model_file.is_file():
                    arcname = f"{run_name}/model/{model_file.relative_to(model_dir)}"
                    zf.write(model_file, arcname)

size_mb = RESULT_ZIP.stat().st_size / 1e6
print(f"Arquivo gerado: {RESULT_ZIP}")
print(f"Tamanho: {size_mb:.1f} MB")
print(f"\nO arquivo esta no Google Drive em: {DRIVE_FOLDER}/colab_bert_results.zip")
print("Baixe e execute localmente:")
print("  uv run python scripts/colab_unpack.py colab_bert_results.zip")

In [ ]:
# Download direto pelo navegador (alternativa ao Drive)
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except Exception:
    print("Download direto nao disponivel. Baixe pelo Google Drive.")